Here is the start of the project. First, I import the data 

In [322]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def fetch_titanic_data():
    tarball_path = Path('datasets/titanic.tgz')
    if not tarball_path.is_file():
        Path('datasets').mkdir(parents=True, exist_ok=True)
        url = 'https://homl.info/titanic.tgz'
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as titanic_tarball:
            titanic_tarball.extractall(path='datasets', filter='data')
    return (pd.read_csv(Path('datasets/titanic/test.csv')), pd.read_csv(Path('datasets/titanic/train.csv')))

test_data, train_data = fetch_titanic_data()

Let's have a look at what we are dealing with

In [323]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Cabin column has 77% of the data missing, it is worth getting rid of

In [324]:
train_data = train_data.drop(columns='Cabin')
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


Let's inspect the non-integer columns

In [325]:
print()
print(train_data['Name'].value_counts()) 
print()
print(train_data['Sex'].value_counts()) 
print()
print(train_data['Ticket'].value_counts()) 
print()
print(train_data['Embarked'].value_counts()) 



Name
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
Allen, Mr. William Henry                               1
                                                      ..
Montvila, Rev. Juozas                                  1
Graham, Miss. Margaret Edith                           1
Johnston, Miss. Catherine Helen 'Carrie'               1
Behr, Mr. Karl Howell                                  1
Dooley, Mr. Patrick                                    1
Name: count, Length: 891, dtype: int64

Sex
male      577
female    314
Name: count, dtype: int64

Ticket
347082             7
1601               7
CA. 2343           7
3101295            6
CA 2144            6
                  ..
SOTON/OQ 392076    1
211536             1
112053             1
111369             1
370376             1
Name: count, Length: 681, dtyp

- Names are unique strings, not sure how a name would be useful for predicting survivability, but will leave it for now
- Sex is a binary value, which would be useful for converting into 0 and 1
- Ticket value is also a (mostly) unique string, although some consist of integers. This column may be useful, but I don't know how to adapt it for the algorithm
- Embarked is a categorical value, that needs to be converted into 1, 2 and 3 or into a one hot

First, lets convert sex from strings to numbers

In [326]:
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder()
sex_cat_encoded = one_hot_encoder.fit_transform(train_data[['Sex']])
train_data[['Sex']] = sex_cat_encoded

Now, the embarked column. Since values there are not related to each other, I will use one hot encoding. Also, because there are 2 instances where the embarked value is missing, I believe deleting them wouldn't be a huge problem for the sake of creating a one hot encoder with 3 values and not 4 that accounts for the missing data

In [327]:
from sklearn.preprocessing import OneHotEncoder
train_data.dropna(subset=['Embarked'], inplace=True)

one_hot_encoder2 = OneHotEncoder()
embarked_cat_encoded = one_hot_encoder2.fit_transform(train_data[['Embarked']])
train_data[['Embarked']] = embarked_cat_encoded

Let's have a look at some other features, that may be categorical

In [328]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    int64  
 1   Survived     889 non-null    int64  
 2   Pclass       889 non-null    int64  
 3   Name         889 non-null    str    
 4   Sex          889 non-null    object 
 5   Age          712 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    str    
 9   Fare         889 non-null    float64
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(2), str(2)
memory usage: 83.3+ KB


In [329]:
print()
print(train_data['Survived'].value_counts()) 
print()
print(train_data['Pclass'].value_counts()) 
print()
print(train_data['SibSp'].value_counts()) 
print()
print(train_data['Parch'].value_counts()) 
print()
print(train_data['Fare'].value_counts()) 


Survived
0    549
1    340
Name: count, dtype: int64

Pclass
3    491
1    214
2    184
Name: count, dtype: int64

SibSp
0    606
1    209
2     28
4     18
3     16
8      7
5      5
Name: count, dtype: int64

Parch
0    676
1    118
2     80
5      5
3      5
4      4
6      1
Name: count, dtype: int64

Fare
8.0500     43
13.0000    42
7.8958     38
7.7500     34
26.0000    31
           ..
13.8583     1
50.4958     1
5.0000      1
9.8458      1
10.5167     1
Name: count, Length: 247, dtype: int64


Survived and Passenger class, would benefit from One Hot encoding.

Fare would benefit from standardisation, because pretty much everything above 50 on the 500 scale cound be considered an outlier, and standardisation is not affected by it
Age graph already looks bell-shaped, so normalisation instead of standardisation would be useful. There is however a considerable number of people aged between 0 and 10, but I don't really know what to do with it, so I'll ignore it for now

Before standardisation and normalisation, fare would also benefit from log transformation 

Passenger ID would need to be normalised, because the range is too big compared to other variables

In [330]:
one_hot_encoder3 = OneHotEncoder()
survived_one_hot_encoded = one_hot_encoder3.fit_transform(train_data[['Survived']])
train_data[['Survived']] = survived_one_hot_encoded

one_hot_encoder4 = OneHotEncoder()
p_class_one_hot_encoded = one_hot_encoder4.fit_transform(train_data[['Pclass']])
train_data[['Pclass']] = p_class_one_hot_encoded

In [331]:
from sklearn.preprocessing import StandardScaler
import numpy as np

train_data["Fare"] = np.log1p(train_data["Fare"])

std_scaler = StandardScaler()
fare_standardised = std_scaler.fit_transform(train_data[['Fare']])
train_data[['Fare']] = fare_standardised

from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()
age_normalised = min_max_scaler.fit_transform(train_data[['Age']])
train_data[['Age']] = age_normalised

min_max_scaler = MinMaxScaler()
id_normalised = min_max_scaler.fit_transform(train_data[['PassengerId']])
train_data[['PassengerId']] = id_normalised



Let's have a look at the data info again

In [332]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    float64
 1   Survived     889 non-null    object 
 2   Pclass       889 non-null    object 
 3   Name         889 non-null    str    
 4   Sex          889 non-null    object 
 5   Age          712 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    str    
 9   Fare         889 non-null    float64
 10  Embarked     889 non-null    object 
dtypes: float64(3), int64(2), object(4), str(2)
memory usage: 83.3+ KB


One thing that is worth transforming is the Name category. Although the names themselves can be treated as random, one feature that could potentially be extracted is the honorific, i.e. Mr, Ms or Dr. For that, I need to process the data, and create a one hot map

After looking at the names, I noticed that they are logged in a very specific format, namely:
{Surname}, {Honorific}. {The rest of the name}
Which means to extract just the honorific, I need to take whatever is in between the comma and the dot, which is easy

The following titles were extracted: array(['Capt', 'Col', 'Don', 'Dr', 'Jonkheer', 'Lady', 'Major', 'Master', 'Miss', 'Mlle', 'Mme', 'Mr', 'Mrs', 'Ms', 'Rev', 'Sir', 'the Countess'])

And the distribution of honorifics is as follows:

{'Mr': 517,
 'Mrs': 124,
 'Miss': 181,
 'Master': 40,
 'Don': 1,
 'Rev': 6,
 'Dr': 7,
 'Mme': 1,
 'Ms': 1,
 'Major': 2,
 'Lady': 1,
 'Sir': 1,
 'Mlle': 2,
 'Col': 2,
 'Capt': 1,
 'the Countess': 1,
 'Jonkheer': 1}

 Obviously, to keep 1 hot encoder's size less than 17, I would need to combine some of the honorifics into a separate category, e.g. Other. The question is, where to draw the line.

 Since the summ of honorifics that are not Mr, Mrs, Miss or Master is less than 40 (it's 27 actually), I guess it is worth combining them into "other category"

In [333]:
names = train_data["Name"]
honorifics = []
common_honorifics = ['Master', 'Miss', 'Mr', 'Mrs']
for name in names:
    candidate = name[name.index(",") + 2 : name.index(".")]
    if candidate in common_honorifics:
        honorifics.append(name[name.index(",") + 2 : name.index(".")])
    else:
        honorifics.append('Other')

# counter = {}
# for honorific in honorifics:
#     if honorific not in counter.keys():
#         counter[honorific] = 1
#     else:
#         counter[honorific] += 1

# counter

honorifics = np.array(honorifics)


train_data["Name"] = honorifics
name_onehot_enc = OneHotEncoder()

name_encoded = name_onehot_enc.fit_transform(train_data[['Name']])
train_data[['Name']] = name_encoded

# name_onehot_enc.categories_
train_data['Name']




0      <Compressed Sparse Row sparse matrix of dtype ...
1      <Compressed Sparse Row sparse matrix of dtype ...
2      <Compressed Sparse Row sparse matrix of dtype ...
3      <Compressed Sparse Row sparse matrix of dtype ...
4      <Compressed Sparse Row sparse matrix of dtype ...
                             ...                        
886    <Compressed Sparse Row sparse matrix of dtype ...
887    <Compressed Sparse Row sparse matrix of dtype ...
888    <Compressed Sparse Row sparse matrix of dtype ...
889    <Compressed Sparse Row sparse matrix of dtype ...
890    <Compressed Sparse Row sparse matrix of dtype ...
Name: Name, Length: 889, dtype: object

When it comes to ticket information though, because the ticket label seems to be random, it is best to drop this category altogether

In [334]:
train_data = train_data.drop(columns='Ticket')
train_data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    float64
 1   Survived     889 non-null    object 
 2   Pclass       889 non-null    object 
 3   Name         889 non-null    object 
 4   Sex          889 non-null    object 
 5   Age          712 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Fare         889 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(3), int64(2), object(5)
memory usage: 76.4+ KB


Now it seems that the last aspect of preprocessing that needs to be addressed is missing entries of the Age category. For simplicity purpose, I will use the median values of this category

In [335]:
median = train_data["Age"].median()
train_data["Age"] = train_data["Age"].fillna(median)
train_data.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    float64
 1   Survived     889 non-null    object 
 2   Pclass       889 non-null    object 
 3   Name         889 non-null    object 
 4   Sex          889 non-null    object 
 5   Age          889 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Fare         889 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(3), int64(2), object(5)
memory usage: 76.4+ KB
